In [ ]:
# push test
# [패키지 설치] 실행 전 터미널에서 아래 명령어를 꼭 실행하세요
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv

import os
import base64
from typing import Optional

# FastAPI 및 서버 관련 라이브러리
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

# AI 모델 및 환경 설정 라이브러리
import ollama
from openai import OpenAI
from dotenv import load_dotenv

# 1. 환경 변수 로드 (.env 파일의 설정을 가져옵니다)
load_dotenv()

app = FastAPI()

# 2. CORS 설정 (모든 허용 - 교안 준수)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def getGptResponse(imageBase64: str, promptText: str) -> str:
    """OpenAI GPT-4o 모델을 사용하여 이미지를 분석합니다."""
    try:
        apiKey = os.getenv("OPENAI_API_KEY")
        client = OpenAI(api_key=apiKey)
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": promptText},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{imageBase64}"}
                        }
                    ]
                }
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"GPT 호출 중 오류 발생: {str(e)}"

def getOllamaResponse(imageBytes: bytes, promptText: str) -> str:
    """로컬 Ollama(gemma4:e2b) 모델을 사용하여 이미지를 분석합니다."""
    try:
        # 모델명은 .env에서 가져오거나 기본값 사용
        targetModel = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
        
        response = ollama.generate(
            model=targetModel,
            prompt=promptText,
            images=[imageBytes]
        )
        return response.get('response', '결과가 없습니다.')
    except Exception as e:
        return f"Ollama 호출 중 오류 발생: {str(e)}"

@app.post("/analyze")
async def analyzeImage(prompt: str = Form(...), file: UploadFile = File(...)):
    """이미지 업로드 및 분석 엔드포인트"""
    try:
        # 파일 데이터 읽기
        imgData = await file.read()
        
        # 모델 선택 확인 (OLLAMA 또는 GPT)
        selectedModel = os.getenv("USE_MODEL", "OLLAMA")
        finalResult = ""

        # 모델별 분기 처리 (명시적 조건문 사용)
        if selectedModel == "GPT":
            # GPT는 base64 문자열이 필요함
            encodedImg = base64.b64encode(imgData).decode('utf-8')
            finalResult = getGptResponse(encodedImg, prompt)
            
        elif selectedModel == "OLLAMA":
            # Ollama는 바이트 데이터를 직접 사용
            finalResult = getOllamaResponse(imgData, prompt)
            
        else:
            finalResult = "설정된 모델이 올바르지 않습니다. (USE_MODEL 확인 필요)"

        # 결과 텍스트 분석 (명시적 반복문 사용 - 코딩 규칙 준수)
        resultWords = finalResult.split()
        lengthList = []
        for idx in range(0, len(resultWords)):
            wordLength = len(resultWords[idx])
            lengthList.append(wordLength)

        return {
            "success": True,
            "message": "분석 완료",
            "data": {
                "activeModel": selectedModel,
                "analysis": finalResult
            }
        }

    except Exception as error:
        # 에러 발생 시 규정된 JSON 형식 반환
        return {
            "success": False,
            "message": f"서버 내부 오류: {str(error)}"
        }

# 3. 서버 실행 설정
if __name__ == "__main__":
    # 주피터 노트북 환경에서도 실행 가능하도록 설정
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve() # ipynb에서는 await를 사용해야 할 수도 있습니다.

INFO:     Started server process [25128]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:65192 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:65192 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:54124 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:54124 - "GET /openapi.json HTTP/1.1" 200 OK
